# 2.1 MuduoXinyu Attention 调优

## 前置要求

具备 C++、ACLNN、模型推理和基础性能统计经验；准备一次性的 MuduoXinyu 基线工作副本，以及同一套模型与 tokenizer 资产。

## 章节目标

- 跑通 FP32 多算子实现包与 FP16 FlashAttentionV4 实现包；
- 用 token exact、fallback 和调用计数判断功能；
- 用 1 次预热 + 3 次统计判断数据是否有效和本次性能是否受益；
- 区分整体实现包结论与纯单变量算法结论。

本实验只比较 Attention 实现，不把量化作为独立实验；也不预设融合路径一定更快。

<img src="images/attention_path_ab.svg" width="900" style="display:block; margin-left:0;" />


## 比较合同

<table style="text-align:left; margin-left:0;">
<tr><th>类别</th><th>项目</th><th>处理</th></tr>
<tr><td>固定条件</td><td>模型、tokenizer、prompt、steps、temperature、topP、Device、统计范围</td><td>A/B 保持一致</td></tr>
<tr><td>实现包 A</td><td>算子组织与 dtype</td><td>FP32 多算子 Attention</td></tr>
<tr><td>实现包 B</td><td>算子组织与 dtype</td><td>FP16 <code>aclnnIncreFlashAttentionV4</code>，含必要 Cast</td></tr>
<tr><td>功能门</td><td>token exact、marker、fallback、调用计数</td><td>必须全部通过</td></tr>
<tr><td>性能门</td><td>1 次预热 + 3 次统计、mean、CV</td><td>数据有效后再判断收益</td></tr>
</table>

A/B 同时改变算子组织和 dtype，因此不是“只改变是否融合”的纯单变量实验。可归因结论只能写“实现包 B 相对实现包 A 在当前 workload 下更快/更慢/无有效结论”，不能把差异单独归因于融合或精度。


## 关键实现

<table style="text-align:left; margin-left:0;">
<tr><th>文件</th><th>作用</th></tr>
<tr><td><code>apply_patch.sh</code></td><td>核对精确基线、补丁哈希和干净工作树；支持 check-only；不 reset/clean/checkout</td></tr>
<tr><td><code>run_ab_benchmark.sh</code></td><td>用固定输入运行 smoke A→B、performance B→A，记录命令、退出码和环境清单</td></tr>
<tr><td><code>analyze_results.py</code></td><td>检查 token exact、fallback 和调用次数，并计算 mean、CV 与性能差异</td></tr>
</table>


## 章节内容

<table style="text-align:left; margin-left:0;">
<tr><th>小节</th><th>内容</th><th>入口</th></tr>
<tr><td>2.2</td><td>环境变量配置、补丁预检、非破坏式构建和实现包 A/B</td><td><a href="./02.02_multi_operator_vs_flash_attention.ipynb">02.02_multi_operator_vs_flash_attention.ipynb</a></td></tr>
<tr><td>2.3</td><td>客观题与简单/中等/困难三档实践</td><td><a href="./02.03_chapter_test.ipynb">02.03_chapter_test.ipynb</a></td></tr>
</table>
